# Aula 06 — Laboratório
## O PIB de hoje, antes do IBGE

**Análise Econômica e Machine Learning** · Prof. Dr. Flavius Sodré · Economia, 8º período

O IBGE divulga o PIB de um trimestre cerca de dois meses depois que ele termina. Antes
disso, a indústria, o varejo, os serviços e o IBC-Br já saíram, mês a mês. Neste
laboratório:

1. do mês ao trimestre: a agregação que o IBGE usa;
2. os modelos que só olham o passado do PIB — média, passeio aleatório e ARIMA;
3. a equação-ponte, com os meses do trimestre que já saíram;
4. Lasso, floresta e boosting com dez indicadores mensais;
5. o nowcast de agora: o PIB do trimestre corrente e o IPCA do mês, com o IPCA-15.

Todas as previsões são **fora da amostra**: para cada trimestre de 2012 em diante, o
modelo é estimado só com o que veio antes dele.

**Como completar:** onde houver `# COMPLETE`, escreva o código. Onde houver `___`,
substitua. Rode as células na ordem.

## Bloco 0 — ambiente

In [ ]:
# Bloco 0 — rode esta celula e confira a saida.
import os, sys, platform, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 120)
plt.rcParams["figure.figsize"] = (10, 4.2)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# As mesmas cores dos slides. Laranja = so o passado do PIB (media, ARIMA).
# Azul = ponte e indicadores mensais. Verde = floresta e boosting.
AZUL, LARANJA, VERDE, BORDO, CINZA = "#2a78d6", "#eb6834", "#1baf7a", "#6F1D2B", "#3a3d45"

try:
    import sklearn
    from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
    from sklearn.linear_model import LassoCV
    from sklearn.model_selection import TimeSeriesSplit
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
except ImportError:
    raise ImportError("Este laboratorio precisa do scikit-learn. "
                      "Instale com:  pip install scikit-learn")

try:
    from statsmodels.tsa.arima.model import ARIMA
    import statsmodels
    VERSAO_SM = statsmodels.__version__
except ImportError:
    VERSAO_SM = "ausente: o ARIMA vira um AR(p) por minimos quadrados"

    class ARIMA:
        """Substituto minimo do ARIMA: AR(p) com constante, por minimos
        quadrados. Ignora a parte MA e a sazonalidade."""
        def __init__(self, y, order=(1, 0, 0), seasonal_order=None):
            self.y, self.p = np.asarray(y, dtype=float), order[0]

        def fit(self):
            return self

        def forecast(self, h=1):
            y, p = self.y, self.p
            if p == 0:
                return np.array([y.mean()])
            X = np.column_stack([np.ones(len(y) - p)] +
                                [y[p - j - 1:len(y) - j - 1] for j in range(p)])
            b = np.linalg.lstsq(X, y[p:], rcond=None)[0]
            return np.array([b[0] + np.dot(b[1:], y[::-1][:p])])

print("Python      :", sys.version.split()[0], "|", platform.system())
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)
print("statsmodels :", VERSAO_SM)
print("\nAmbiente pronto.")

## Bloco 1 — do mês ao trimestre

Duas tabelas. A **mensal** tem o IBC-Br, sete recortes da produção industrial (PIM-PF),
o varejo e o varejo ampliado (PMC) e os serviços (PMS), todos índices com ajuste sazonal,
mais o IPCA e o IPCA-15 em variação no mês. A **trimestral** tem o índice do PIB com
ajuste sazonal (IBGE, Contas Nacionais Trimestrais).

In [ ]:
# Bloco 1 — Plano B. Rode e siga em frente; nao precisa ler.
#
# Resumo da base real (comeco e fim de cada serie, nivel inicial, quanto cada
# indicador acompanha a atividade, a ponte do PIB e o perfil do IPCA). Serve
# so para gerar dados inventados com a mesma estrutura.

INDICADORES = ['ibcbr', 'pim_geral', 'pim_extrativa', 'pim_transformacao', 'pim_capital', 'pim_intermediarios', 'pim_duraveis', 'pim_semi_nao_duraveis', 'varejo', 'varejo_ampliado']
COLUNAS_MENSAL = INDICADORES + ["servicos", "ipca", "ipca15"]
NOMES = {'ibcbr': 'IBC-Br',
 'pim_geral': 'Indústria geral',
 'pim_extrativa': 'Indústria extrativa',
 'pim_transformacao': 'Indústria de transformação',
 'pim_capital': 'Bens de capital',
 'pim_intermediarios': 'Bens intermediários',
 'pim_duraveis': 'Bens de consumo duráveis',
 'pim_semi_nao_duraveis': 'Semi e não duráveis',
 'varejo': 'Varejo',
 'varejo_ampliado': 'Varejo ampliado',
 'servicos': 'Serviços',
 'ipca': 'IPCA',
 'ipca15': 'IPCA-15'}

# Avaliacao fora da amostra a partir de 2012T1; os dois trimestres do choque.
INICIO_AVAL = pd.Period("2012Q1", "Q")
CHOQUE = (pd.Period("2020Q2", "Q"), pd.Period("2020Q3", "Q"))

ESP_PLANO_B = {'series': {'ibcbr': {'inicio': '2003-01',
                      'fim': '2026-08',
                      'nivel': 70.25,
                      'beta': 1.0,
                      'ruido': 0.0},
            'pim_geral': {'inicio': '2002-01',
                          'fim': '2026-08',
                          'nivel': 90.77,
                          'beta': 1.275,
                          'ruido': 0.0144},
            'pim_extrativa': {'inicio': '2002-01',
                              'fim': '2026-08',
                              'nivel': 74.87,
                              'beta': 1.045,
                              'ruido': 0.0292},
            'pim_transformacao': {'inicio': '2002-01',
                                  'fim': '2026-08',
                                  'nivel': 91.81,
                                  'beta': 1.311,
                                  'ruido': 0.0164},
            'pim_capital': {'inicio': '2002-01',
                            'fim': '2026-08',
                            'nivel': 63.86,
                            'beta': 2.158,
                            'ruido': 0.0507},
            'pim_intermediarios': {'inicio': '2002-01',
                                   'fim': '2026-08',
                                   'nivel': 94.27,
                                   'beta': 1.042,
                                   'ruido': 0.0137},
            'pim_duraveis': {'inicio': '2002-01',
                             'fim': '2026-08',
                             'nivel': 84.17,
                             'beta': 3.04,
                             'ruido': 0.0525},
            'pim_semi_nao_duraveis': {'inicio': '2002-01',
                                      'fim': '2026-08',
                                      'nivel': 95.39,
                                      'beta': 1.234,
                                      'ruido': 0.0192},
            'varejo': {'inicio': '2000-01',
                       'fim': '2026-08',
                       'nivel': 50.14,
                       'beta': 0.385,
                       'ruido': 0.0121},
            'varejo_ampliado': {'inicio': '2003-01',
                                'fim': '2026-08',
                                'nivel': 47.19,
                                'beta': 1.071,
                                'ruido': 0.0208},
            'servicos': {'inicio': '2011-01',
                         'fim': '2026-08',
                         'nivel': 89.01,
                         'beta': 0.895,
                         'ruido': 0.0102}},
 'atividade': {'media': 0.00185, 'dp': 0.00839, 'ar': 0.046},
 'pib': {'inicio': '1996Q1',
         'fim': '2026Q2',
         'nivel': 99.41,
         'a': 0.179,
         'b': 0.807,
         'ruido': 0.491},
 'ipca': {'inicio': '2000-01',
          'fim': '2026-09',
          'inicio15': '2000-05',
          'fim15': '2026-10',
          'perfil': [0.56, 0.67, 0.576, 0.493, 0.415, 0.301, 0.277, 0.197, 0.381, 0.498, 0.48,
                     0.628],
          'ar': 0.497,
          'dp': 0.258,
          'ponte15': [-0.028, 0.603, 0.458],
          'ruido15': 0.114}}

CHOQUES_PLANO_B = {'2008-11': -0.025,
 '2008-12': -0.03,
 '2009-01': 0.01,
 '2020-03': -0.07,
 '2020-04': -0.1,
 '2020-05': 0.03,
 '2020-06': 0.045,
 '2020-07': 0.027,
 '2020-08': 0.02,
 '2020-09': 0.02}


def crescimento(serie):
    """Variação percentual contra o período anterior."""
    return 100 * (serie / serie.shift(1) - 1)


def dados_sinteticos(ESP, semente=2026):
    """Plano B: indicadores e PIB INVENTADOS, com as mesmas colunas, os mesmos
    meses e trimestres e as mesmas relações médias da base real. A atividade
    mensal é um processo com memória, com as crises de 2008, 2015–2016 e
    2020; cada indicador acompanha a atividade com sensibilidade e ruído
    próprios; o PIB segue a média trimestral da atividade."""
    rng = np.random.default_rng(semente)
    meses = pd.period_range("1996-01", ESP["ipca"]["fim15"], freq="M")
    a = ESP["atividade"]
    g = np.zeros(len(meses))
    for i in range(1, len(meses)):
        g[i] = a["media"] + a["ar"] * (g[i - 1] - a["media"]) + a["dp"] * rng.normal()
    for i, m in enumerate(meses):
        if pd.Period("2015-01", "M") <= m <= pd.Period("2016-12", "M"):
            g[i] -= 0.004
        g[i] += CHOQUES_PLANO_B.get(str(m), 0.0)
    nivel = np.cumsum(g)
    mensal = pd.DataFrame(index=meses)
    for c, e in ESP["series"].items():
        ruido = np.cumsum(e["ruido"] * rng.normal(size=len(meses)) * 0.6)
        v = np.exp(e["beta"] * nivel + ruido)
        dentro = (meses >= pd.Period(e["inicio"], "M")) & (meses <= pd.Period(e["fim"], "M"))
        v = v / v[dentro][0] * e["nivel"]
        mensal[c] = np.where(dentro, np.round(v, 2), np.nan)
    p = ESP["ipca"]
    desvio = np.zeros(len(meses))
    for i in range(1, len(meses)):
        desvio[i] = p["ar"] * desvio[i - 1] + p["dp"] * rng.normal()
    ipca = np.array(p["perfil"])[meses.month - 1] + desvio
    i15 = (p["ponte15"][0] + p["ponte15"][1] * ipca + p["ponte15"][2] * np.r_[ipca[0], ipca[:-1]]
           + p["ruido15"] * rng.normal(size=len(meses)))
    mensal["ipca"] = np.where((meses >= pd.Period(p["inicio"], "M")) &
                              (meses <= pd.Period(p["fim"], "M")), np.round(ipca, 2), np.nan)
    mensal["ipca15"] = np.where((meses >= pd.Period(p["inicio15"], "M")) &
                                (meses <= pd.Period(p["fim15"], "M")), np.round(i15, 2), np.nan)
    mensal = mensal[mensal.index >= pd.Period("2000-01", "M")][COLUNAS_MENSAL]
    ativ = pd.Series(np.exp(nivel), index=meses)
    tri = ativ.groupby(ativ.index.asfreq("Q")).mean()
    q = ESP["pib"]
    trimestres = pd.period_range(q["inicio"], q["fim"], freq="Q")
    gq = crescimento(tri).reindex(trimestres)
    cres = q["a"] + q["b"] * gq + q["ruido"] * rng.normal(size=len(trimestres))
    cres.iloc[0] = 0.0
    pib_nivel = q["nivel"] * np.cumprod(1 + cres.to_numpy() / 100)
    trimestral = pd.DataFrame({"pib": np.round(pib_nivel, 2)}, index=trimestres)
    return mensal, trimestral


print("Plano B pronto (so e usado se os dados reais nao carregarem).")

In [ ]:
# Bloco 1 — os dados. Tres caminhos, nesta ordem:
#   1. os arquivos indicadores_mensais.csv e pib_trimestral.csv, na pasta
#      Dados do repositorio;
#   2. as APIs do Banco Central (SGS) e do IBGE (SIDRA) — lentas: ate 2 min;
#   3. o PLANO B (celula acima), com dados inventados.
# Cada chamada tenta de novo desde o comeco: se a rede voltar, basta rodar
# a celula outra vez.
import requests

ARQ_MENSAL, ARQ_TRIMESTRAL = "../Dados/indicadores_mensais.csv", "../Dados/pib_trimestral.csv"
SERIES_SGS = {"ibcbr": 24364, "ipca": 433, "ipca15": 7478}
SERIES_SIDRA = {
    "pim_geral": "t/8888/n1/all/v/12607/p/all/c544/129314",
    "pim_extrativa": "t/8888/n1/all/v/12607/p/all/c544/129315",
    "pim_transformacao": "t/8888/n1/all/v/12607/p/all/c544/129316",
    "pim_capital": "t/8887/n1/all/v/12607/p/all/c543/129278",
    "pim_intermediarios": "t/8887/n1/all/v/12607/p/all/c543/129283",
    "pim_duraveis": "t/8887/n1/all/v/12607/p/all/c543/129301",
    "pim_semi_nao_duraveis": "t/8887/n1/all/v/12607/p/all/c543/129305",
    "varejo": "t/8880/n1/all/v/7170/p/all/c11046/56734",
    "varejo_ampliado": "t/8881/n1/all/v/7170/p/all/c11046/56736",
    "servicos": "t/5906/n1/all/v/7168/p/all/c11046/56726",
}
PIB_SIDRA = "t/1621/n1/all/v/584/p/all/c11255/90707"   # PIB com ajuste sazonal


def ler_csv(arquivo_mensal, arquivo_trimestral):
    """Lê as duas tabelas com o período como índice (mês e trimestre)."""
    mensal = pd.read_csv(arquivo_mensal)
    mensal.index = pd.PeriodIndex(mensal.pop("mes"), freq="M")
    trimestral = pd.read_csv(arquivo_trimestral)
    trimestral.index = pd.PeriodIndex(trimestral.pop("trimestre"), freq="Q")
    return mensal, trimestral


def baixar_sgs(codigo, timeout=120):
    """Serie mensal do SGS do Banco Central, com o mes como indice."""
    hoje = pd.Timestamp.today().strftime("%d/%m/%Y")
    url = (f"https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"
           f"?formato=json&dataInicial=01/01/2000&dataFinal={hoje}")
    r = requests.get(url, timeout=timeout)
    r.raise_for_status()
    j = r.json()
    meses = pd.PeriodIndex([d["data"][6:] + "-" + d["data"][3:5] for d in j], freq="M")
    return pd.Series([float(d["valor"]) for d in j], index=meses)


def baixar_sidra(caminho, trimestral=False, timeout=120):
    """Serie do SIDRA/IBGE. A primeira linha da resposta e o cabecalho; o
    periodo vem em D3C (202608 = ago/2026; 202602 = 2026, 2o trimestre)."""
    r = requests.get("https://apisidra.ibge.gov.br/values/" + caminho, timeout=timeout,
                     headers={"Accept": "application/json"})
    r.raise_for_status()
    linhas = r.json()[1:]
    cod = [d["D3C"] for d in linhas]
    if trimestral:
        idx = pd.PeriodIndex([c[:4] + "Q" + str(int(c[4:])) for c in cod], freq="Q")
    else:
        idx = pd.PeriodIndex([c[:4] + "-" + c[4:] for c in cod], freq="M")
    valores = pd.to_numeric(pd.Series([d["V"] for d in linhas]), errors="coerce").to_numpy()
    return pd.Series(valores, index=idx).round(2)


def baixar_tudo():
    series = {n: baixar_sgs(c) for n, c in SERIES_SGS.items()}
    series.update({n: baixar_sidra(c) for n, c in SERIES_SIDRA.items()})
    mensal = pd.DataFrame(series)
    mensal = mensal.reindex(pd.period_range("2000-01", mensal.index.max(), freq="M"))
    trimestral = baixar_sidra(PIB_SIDRA, trimestral=True).rename("pib").to_frame()
    return mensal[COLUNAS_MENSAL], trimestral


def carregar_dados():
    if os.path.exists(ARQ_MENSAL) and os.path.exists(ARQ_TRIMESTRAL):
        return (*ler_csv(ARQ_MENSAL, ARQ_TRIMESTRAL), "arquivos locais")
    try:
        return (*baixar_tudo(), "APIs do Banco Central e do IBGE")
    except Exception as erro:
        print("download falhou:", type(erro).__name__, str(erro)[:80])
    print("=" * 70)
    print("PLANO B: os dados abaixo sao INVENTADOS. Mesmas colunas e mesmos")
    print("meses da base real; os valores sao sinteticos. Os numeros NAO")
    print("valem como resultado sobre a economia brasileira.")
    print("=" * 70)
    return (*dados_sinteticos(ESP_PLANO_B), "PLANO B (sintetico)")


mensal, trimestral, FONTE = carregar_dados()
print(f"fonte: {FONTE}")
print(f"mensal    : {mensal.index[0]} a {mensal.index[-1]}, {mensal.shape[1]} colunas")
print(f"trimestral: {trimestral.index[0]} a {trimestral.index[-1]}")
print("\nultimo mes disponivel de cada serie:")
print(mensal.apply(lambda s: s.dropna().index[-1]).to_string())

In [ ]:
# Bloco 1 — as ferramentas. Leia, nao precisa alterar.

def media_trimestral(serie_mensal):
    """Média dos três meses de cada trimestre; só trimestres completos."""
    s = serie_mensal.dropna()
    g = s.groupby(s.index.asfreq("Q"))
    return g.mean()[g.size() == 3]


def indicador_trimestral(serie_mensal):
    """Crescimento % da média do trimestre contra a média do trimestre
    anterior: é assim que o IBGE compara um trimestre com o outro."""
    return crescimento(media_trimestral(serie_mensal))


def matriz_trimestral(mensal, colunas):
    """Uma coluna por indicador: o crescimento da média de cada trimestre."""
    return pd.concat([indicador_trimestral(mensal[c]).rename(c) for c in colunas], axis=1)


def indicador_com_k_meses(serie_mensal, trimestre, k):
    """O mesmo crescimento, quando só os k primeiros meses do trimestre já
    saíram. Os meses que faltam repetem o último valor conhecido; com k = 0,
    nenhum mês saiu, e o resultado é o carregamento estatístico."""
    s = serie_mensal.dropna()
    meses = pd.period_range(trimestre.asfreq("M", "start"), periods=3, freq="M")
    conhecidos = s[s.index < meses[0]] if k == 0 else s[s.index <= meses[k - 1]]
    if k > 0 and conhecidos.index[-1] != meses[k - 1]:
        return np.nan                      # o mês k ainda não saiu
    valores = list(conhecidos[conhecidos.index >= meses[0]])
    valores += [conhecidos.iloc[-1]] * (3 - len(valores))
    anterior = s[(s.index >= meses[0] - 3) & (s.index < meses[0])]
    if len(anterior) < 3:
        return np.nan
    return float(100 * (np.mean(valores) / anterior.mean() - 1))


def janela_expansiva(prever, alvo):
    """Para cada período t do alvo, o modelo é estimado só com o que veio
    antes de t e prevê t. Devolve a série das previsões."""
    return pd.Series({t: prever(t) for t in alvo}, dtype=float)


def rmse(previsoes, realizado, sem_choque=False):
    """Raiz do erro quadrático médio, em pontos percentuais. Com
    `sem_choque`, deixa de fora 2020T2 e 2020T3."""
    e = (previsoes - realizado.reindex(previsoes.index)).dropna()
    if sem_choque:
        e = e.drop([t for t in CHOQUE if t in e.index])
    return float(np.sqrt(np.mean(e ** 2)))


print("Ferramentas prontas.")

In [ ]:
# Bloco 1 — o alvo. COMPLETE.
#
# O alvo e o crescimento do PIB contra o trimestre anterior, em %.

# COMPLETE
y = ___

print(f"{len(y)} trimestres, {y.index[0]} a {y.index[-1]}")
print(f"media {y.mean():.2f}% | menor {y.min():.2f}% em {y.idxmin()} | "
      f"maior {y.max():.2f}% em {y.idxmax()}")
fig, ax = plt.subplots()
ax.bar(range(len(y)), y, color=[AZUL if v >= 0 else LARANJA for v in y])
ax.set_xticks(range(0, len(y), 12)); ax.set_xticklabels([str(t) for t in y.index[::12]])
ax.set_ylabel("% contra o trimestre anterior")
plt.show()

In [ ]:
# Bloco 1 — a agregacao, a mao. COMPLETE.
#
# O IBGE compara a MEDIA dos tres meses de um trimestre com a media dos tres
# meses do anterior. Faca a conta para o IBC-Br no 2o trimestre de 2026 e
# confira com a funcao indicador_trimestral.
ibc = mensal["ibcbr"]

# COMPLETE
media_t1 = ___      # media de jan, fev e mar de 2026
media_t2 = ___      # media de abr, mai e jun de 2026
print(f"1o tri {media_t1:.2f} | 2o tri {media_t2:.2f} | "
      f"crescimento {100 * (media_t2 / media_t1 - 1):.2f}%")
print(f"pela funcao: {indicador_trimestral(ibc)[pd.Period('2026Q2', 'Q')]:.2f}%")

In [ ]:
# Bloco 1 — a matriz trimestral e a correlacao. COMPLETE.
#
# Uma coluna por indicador (o crescimento da media de cada trimestre) e a
# correlacao de cada coluna com o crescimento do PIB.

# COMPLETE
X = ___
correlacao = ___

print(correlacao.sort_values(ascending=False).round(2))
tabela = pd.concat([y.rename("pib"), X], axis=1).dropna()
print(f"\ncom todos os dez indicadores: {len(tabela)} trimestres, "
      f"{tabela.index[0]} a {tabela.index[-1]}")

## Bloco 2 — o passado do PIB contra a média

Para cada trimestre t de 2012T1 em diante, o modelo vê só o crescimento **até t − 1** e
prevê t. É a janela expansiva.

In [ ]:
# Bloco 2 — o ARIMA. Leia, nao precisa alterar.

def prever_arima(y, t, ordem=(1, 0, 0)):
    """ARIMA estimado no crescimento até o trimestre anterior a t. Modelar o
    crescimento já é diferenciar o nível: ARIMA(p, 0, q) no crescimento é
    ARIMA(p, 1, q) no nível do PIB."""
    h = y[y.index < t].dropna().to_numpy()
    with warnings.catch_warnings():        # avisos de convergência do statsmodels
        warnings.simplefilter("ignore")
        return float(ARIMA(h, order=ordem).fit().forecast(1)[0])


print("ARIMA pronto.")

In [ ]:
# Bloco 2 — as duas linhas de base. COMPLETE.

def prever_media(y, t):
    """Media historica do crescimento ate o trimestre anterior a t."""
    # COMPLETE
    return ___


def prever_passeio(y, t):
    """Passeio aleatorio: repete o crescimento do trimestre anterior."""
    # COMPLETE
    return ___


alvo = [t for t in y.index if t >= INICIO_AVAL]
prev = {
    "media": janela_expansiva(lambda t: prever_media(y, t), alvo),
    "passeio": janela_expansiva(lambda t: prever_passeio(y, t), alvo),
    "arima": janela_expansiva(lambda t: prever_arima(y, t), alvo),
}
erros = pd.DataFrame({n: {"com o choque": rmse(p, y), "sem o choque": rmse(p, y, True)}
                      for n, p in prev.items()}).T
print(f"{len(alvo)} trimestres, {alvo[0]} a {alvo[-1]}\n")
print(erros.round(2))

In [ ]:
# Bloco 2 — outras ordens. COMPLETE.
#
# ARIMA(p, 0, q) para p e q de 0 a 2: o RMSE com e sem o choque. Leva cerca
# de um minuto.
linhas = []
for p in range(3):
    for q in range(3):
        # COMPLETE
        pr = ___
        linhas.append({"p": p, "q": q, "com o choque": rmse(pr, y),
                       "sem o choque": rmse(pr, y, True)})
grade = pd.DataFrame(linhas)
print(grade.round(2))
print(f"\nmedia historica, sem o choque: {rmse(prev['media'], y, True):.2f}")

## Bloco 3 — a ponte, mês a mês

A equação-ponte estima, com os trimestres já divulgados, a reta do crescimento do PIB
contra o crescimento de um indicador mensal. Para o trimestre corrente, usa os meses
que já saíram e repete o último até completar o trimestre.

In [ ]:
# Bloco 3 — o ajuste da ponte. Leia, nao precisa alterar.

def ajustar_ponte(y, X):
    """Mínimos quadrados de y contra as colunas de X, com constante.
    Devolve [constante, coeficientes...]."""
    d = pd.concat([y, X], axis=1).dropna()
    A = np.column_stack([np.ones(len(d)), d.iloc[:, 1:].to_numpy()])
    return np.linalg.lstsq(A, d.iloc[:, 0].to_numpy(), rcond=None)[0]


print("Ajuste pronto.")

In [ ]:
# Bloco 3 — os k meses de 2020T2. COMPLETE.
#
# O crescimento do IBC-Br no 2o trimestre de 2020 com k = 0, 1, 2 e 3 meses
# ja divulgados. Com k = 0, repete-se marco (o carregamento estatistico).
t20 = pd.Period("2020Q2", "Q")

# COMPLETE
for k in range(4):
    print(f"k = {k}: IBC-Br {___:.2f}%")

In [ ]:
# Bloco 3 — a ponte. COMPLETE.

def prever_ponte(y, mensal, colunas, t, k):
    """Estima a reta com os trimestres completos ate t - 1 e aplica ao
    trimestre t com os k meses que ja sairam."""
    X = pd.concat([indicador_trimestral(mensal[c]) for c in colunas], axis=1)
    # COMPLETE: coeficientes com o que veio antes de t
    b = ___
    # COMPLETE: o crescimento de cada indicador no trimestre t, com k meses
    x = ___
    return float(b[0] + np.dot(b[1:], x))


for k in range(4):
    prev[f"ponte k={k}"] = janela_expansiva(
        lambda t, k=k: prever_ponte(y, mensal, ["ibcbr"], t, k), alvo)
erros = pd.DataFrame({n: {"com o choque": rmse(p, y), "sem o choque": rmse(p, y, True),
                          "2020T2": p[t20]} for n, p in prev.items()}).T
print(erros.round(2))
print(f"\nrealizado em 2020T2: {y[t20]:.2f}%")

In [ ]:
# Bloco 3 — outra ponte. COMPLETE.
#
# Industria geral e varejo juntos, que saem antes do IBC-Br. Compare com a
# ponte do IBC-Br para k = 1, 2 e 3.

# COMPLETE
___

### ✔ CHECKPOINT I

Para o 2º trimestre de 2020, o ARIMA previu perto de zero; a ponte, só com abril, mais de
10% de queda. O PIB caiu 8,9%.

Em duas frases: por que o modelo de série temporal falha justamente quando a previsão
mais importa? O que a ponte tem que ele não tem?

## Bloco 4 — dez indicadores, três modelos

A mesma janela expansiva, agora com os dez indicadores mensais: Lasso (Aula 04), floresta
e boosting (Aula 05). Os três recebem o crescimento de cada indicador no trimestre.

In [ ]:
# Bloco 4 — os tres modelos. Leia, nao precisa alterar.

def modelo_lasso():
    """Lasso com variáveis padronizadas; o λ sai de uma validação que respeita
    a ordem do tempo (cada dobra valida em trimestres posteriores aos do
    ajuste)."""
    return make_pipeline(StandardScaler(),
                         LassoCV(alphas=np.logspace(-3, 1, 60), cv=TimeSeriesSplit(5),
                                 max_iter=50000))


def modelo_floresta():
    return RandomForestRegressor(n_estimators=300, min_samples_leaf=3, max_features=0.33,
                                 random_state=0, n_jobs=-1)


def modelo_boosting():
    return GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=2,
                                     min_samples_leaf=3, subsample=0.8, random_state=0)


MODELOS_ML = {"lasso": modelo_lasso, "floresta": modelo_floresta, "boosting": modelo_boosting}


def prever_ml(fabrica, y, mensal, colunas, t, k):
    """Mesma lógica da ponte, com muitos indicadores e um modelo de machine
    learning no lugar dos mínimos quadrados."""
    d = pd.concat([y.rename("pib"), matriz_trimestral(mensal, colunas)], axis=1).dropna()
    d = d[d.index < t]
    modelo = fabrica().fit(d[colunas], d["pib"])
    x = pd.DataFrame([[indicador_com_k_meses(mensal[c], t, k) for c in colunas]],
                     columns=colunas)
    return float(modelo.predict(x)[0]), modelo


print("Modelos prontos.")

In [ ]:
# Bloco 4 — os tres modelos na janela expansiva. COMPLETE.
#
# Com os tres meses do trimestre (k = 3). Leva cerca de dois minutos.
prev_ml = {}
for nome, fabrica in MODELOS_ML.items():
    # COMPLETE: previsoes de prever_ml para cada trimestre do alvo
    # (prever_ml devolve a previsao e o modelo; fique so com a previsao)
    prev_ml[nome] = ___

comparacao = pd.DataFrame(
    {n: {"com o choque": rmse(p, y), "sem o choque": rmse(p, y, True), "2020T2": p[t20]}
     for n, p in {"ponte IBC-Br": prev["ponte k=3"], **prev_ml}.items()}).T
print(comparacao.round(2))

In [ ]:
# Bloco 4 — o que o Lasso escolhe e o que a floresta nao ve. COMPLETE.
#
# (a) Lasso com todos os trimestres: quais indicadores ficam?
# (b) Floresta e reta so com o IBC-Br, estimadas ate 2020T1: a previsao para
#     uma grade de crescimentos do IBC-Br de -11% a 4%.
lasso = modelo_lasso().fit(tabela[INDICADORES], tabela["pib"])
print(pd.Series(lasso[-1].coef_, index=INDICADORES).round(3))

treino = tabela[tabela.index <= pd.Period("2020Q1", "Q")]
grade_x = pd.DataFrame({"ibcbr": np.linspace(-11, 4, 151)})
# COMPLETE: a floresta (300 arvores, folhas de 3) e a reta, so com o IBC-Br
floresta_1 = ___
b = ___

fig, ax = plt.subplots()
ax.scatter(treino["ibcbr"], treino["pib"], s=14, color="#8A8D96", label="treino")
ax.plot(grade_x["ibcbr"], b[0] + b[1] * grade_x["ibcbr"], color=AZUL, lw=2.5, label="reta")
ax.plot(grade_x["ibcbr"], floresta_1.predict(grade_x), color=VERDE, lw=2.5, label="floresta")
ax.scatter([X.loc[t20, "ibcbr"]], [y[t20]], s=80, color=BORDO, label="2020T2 realizado")
ax.set_xlabel("IBC-Br, % no trimestre"); ax.set_ylabel("PIB previsto, %")
ax.legend(frameon=False); plt.show()

In [ ]:
# Bloco 4 — sem o IBC-Br. COMPLETE.
#
# Tire o IBC-Br da matriz e rode so o Lasso com os nove indicadores restantes.
sem_ibc = [c for c in INDICADORES if c != "ibcbr"]

# COMPLETE
prev_lasso9 = ___
print(f"Lasso com nove indicadores: {rmse(prev_lasso9, y):.2f} / {rmse(prev_lasso9, y, True):.2f}")

## Bloco 5 — o nowcast de agora

O primeiro trimestre que ainda não está na tabela do PIB é o que o IBGE ainda não
divulgou. Que meses de cada indicador já saíram?

In [ ]:
# Bloco 5 — o PIB do trimestre corrente. COMPLETE.
ATUAL = trimestral["pib"].dropna().index[-1] + 1
meses = pd.period_range(ATUAL.asfreq("M", "start"), periods=3, freq="M")
saidos = {c: int(sum(m in mensal[c].dropna().index for m in meses)) for c in INDICADORES}
k = min(saidos.values())
print(f"trimestre a prever: {ATUAL} | meses ja divulgados: {k}")
print({c: round(indicador_com_k_meses(mensal[c], ATUAL, k), 2) for c in INDICADORES})

# COMPLETE: o nowcast do PIB com a ponte do IBC-Br, com industria + varejo, com
# o Lasso dos dez indicadores e com o ARIMA
nowcast = {
    "ponte IBC-Br": ___,
    "ponte industria + varejo": ___,
    "lasso": ___,
    "arima": ___,
}
print(pd.Series(nowcast).round(2))

In [ ]:
# Bloco 5 — os modelos do IPCA. Leia, nao precisa alterar.

def prever_ipca_passeio(ipca, m):
    return float(ipca[ipca.index < m].iloc[-1])


def prever_ipca_sarima(ipca, m):
    """ARIMA sazonal: o mês anterior e o mesmo mês do ano anterior."""
    h = ipca[ipca.index < m].dropna().to_numpy()
    with warnings.catch_warnings():        # avisos de convergência do statsmodels
        warnings.simplefilter("ignore")
        modelo = ARIMA(h, order=(1, 0, 0), seasonal_order=(1, 0, 0, 12))
        return float(modelo.fit().forecast(1)[0])


def prever_ipca_ponte(ipca, ipca15, m):
    """IPCA do mês contra o IPCA-15 do mesmo mês, estimado até o mês anterior."""
    b = ajustar_ponte(ipca[ipca.index < m], ipca15[ipca15.index < m])
    return float(b[0] + b[1] * ipca15[m])


print("Modelos do IPCA prontos.")

In [ ]:
# Bloco 5 — o IPCA com o IPCA-15. COMPLETE.
#
# De jan/2012 em diante: passeio aleatorio, ARIMA sazonal e a ponte com o
# IPCA-15. Depois, o nowcast do mes cujo IPCA-15 ja saiu e o IPCA nao.
ipca, ipca15 = mensal["ipca"].dropna(), mensal["ipca15"].dropna()
alvo_ipca = [m for m in ipca.index if m >= pd.Period("2012-01", "M")]

# COMPLETE
prev_ipca = {
    "passeio": ___,
    "arima sazonal": ___,
    "ponte IPCA-15": ___,
}
for n, p in prev_ipca.items():
    print(f"{n:<14s} RMSE {rmse(p, ipca):.3f}")

mes = ipca15.index[-1]
if mes not in ipca.index:
    print(f"\n{mes}: IPCA-15 {ipca15[mes]:.2f}% -> IPCA previsto "
          f"{prever_ipca_ponte(ipca, ipca15, mes):.2f}%")

### ✔ CHECKPOINT II

Sem os trimestres do choque, o Lasso com dez indicadores erra praticamente o mesmo que a
ponte só com o IBC-Br.

Em duas frases: isso quer dizer que machine learning não serve para nowcasting? Em que
situação os dez indicadores fariam diferença?

### Responda

> O Banco Central divulga o IBC-Br, que acompanha o PIB de perto. Por que um analista de
> banco ainda montaria a própria ponte com a indústria e o varejo?